# 03 | Model comparison, training and validation

This notebook retains executable baseline fitting, feature construction, statistical/ridge/segment comparisons, recursive VM LightGBM training, chronological validation, count experiments, direct-model ablations and activity-classifier training. LightGBM remains the final forecast in `predictions/prediction.csv`. CPU errors are percentage points; the viewed holdout is not an independent final-model test.

Run from top to bottom. Aggregate comparisons and final-model fitting run normally. The expensive full operational comparison has an explicit refit switch, as in the source experiment, with all training and calibration code retained in the package.

## 1. Prepare the public training window

**Motive:** Keep holdout labels outside baseline selection.

**Logic:** Verify files, construct the full five-minute target grid and reserve the final 24 timestamps. Pass only the preceding training frame to the backtester. Holdout target values are not displayed, scored or used for selection. No random split or shuffling is used.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src/cwfc").exists(): ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
import sys
from pathlib import Path
import os
import json
import logging
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "challenge_data").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "challenge_data").is_dir()
os.environ["MPLCONFIGDIR"] = str(ROOT / "artifacts" / "matplotlib")
import matplotlib.pyplot as plt
from cwfc.io import verify_checksums, load_readings, load_metadata
from cwfc.targets import build_targets, split_train_holdout
from cwfc.validation import backtest_baselines, score_backtest, select_moving_average, summarize_folds
from cwfc.eda import plot_validation_windows, plot_baseline_scores, save_figure

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
ARTIFACTS = ROOT / "artifacts" / "baseline_validation"
FIGURES = ROOT / "reports" / "figures"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
verify_checksums(ROOT / "challenge_data")
readings = load_readings(ROOT / "challenge_data")
metadata = load_metadata(ROOT / "challenge_data")
full_targets = build_targets(readings, metadata)
train_targets, holdout_targets = split_train_holdout(full_targets, min_train=48)
settings = {"horizon": 24, "min_train": 48, "step": 6}
boundary = pd.DataFrame([
    ("Training", len(train_targets), int(train_targets.timestamp.min()), int(train_targets.timestamp.max())),
    ("Reserved holdout: timestamps only", len(holdout_targets), int(holdout_targets.timestamp.min()), int(holdout_targets.timestamp.max())),
], columns=["Window", "Rows", "First relative second", "Last relative second"])
display(boundary.style.hide(axis="index"))


INFO: Excluded 0 incomplete CPU readings (no imputation)
INFO: Averaging 0 duplicate VM/timestamp rows


Window,Rows,First relative second,Last relative second
Training,110,0,32700
Reserved holdout: timestamps only,24,33000,39900


## 2. Define comparable baselines

**Motive:** Test whether recent level, smoothing, a trend, a repeated pattern  provides a useful reference.

Let `y[n−1]` be the final available value, and `h` the forecast step starting at one.

| Candidate | Forecast logic | Parameter rule |
|---|---|---|
| Persistence | Repeat `y[n−1]` | None |
| Moving average | Repeat the mean of the last `k` observations | Compare k = 3, 6, 12, 24 using CV MAE |
| Drift | `y[n−1] + h × (y[n−1] − y[0]) / (n−1)` | Mean first-difference slope over available history |
| Seasonal naive | Repeat the final 24 observations in order | Length 24, fixed negative control |

The seasonal-naive period is **24 five-minute steps (two hours)**. It is a negative control, not evidence of daily seasonality. The trace has relative timestamps and insufficient duration to establish daily repetition.

Apply identical postprocessing to all candidates: CPU forecasts clipped to [0,100], count forecasts floored at zero, with no count rounding. Missing/nonfinite target values cause a clear error; the backtester does not impute targets or silently discard timestamps.


## 3. Move the forecast origin forward

**Motive:** Reproduce a system that repeatedly predicts the next two hours using only observations already available.

**Logic:** Start with 48 observations. At each origin, train on rows `[0, origin)`, forecast the next 24 rows and then advance six rows (30 minutes). The origin is the **number of training rows**, also the first validation index; the latest observed row is `origin − 1`.

Origins follow `range(48, n − 24 + 1, 6)`. For n = 110, the last feasible origin is 84: the next proposed origin, 90, would require 114 observations. Forecast functions receive only past target values and a template containing future timestamps, with validation labels attached after prediction.

Validation windows overlap. A timestamp forecast from different origins represents different forecast decisions; pooled cases are not independent observations. The fixed grid leaves the last two training timestamps outside these validation windows.


In [2]:
backtest = backtest_baselines(train_targets, **settings)
folds = summarize_folds(backtest)
display(folds.rename(columns={"fold":"Fold", "origin":"Origin / history rows", "train_start_seconds":"History starts (s)", "train_end_seconds":"History ends (s)", "validation_start_seconds":"Validation starts (s)", "validation_end_seconds":"Validation ends (s)", "validation_steps":"Forecast steps"})
    .style.hide(axis="index"))
fig = plot_validation_windows(backtest)
save_figure(fig, FIGURES, "validation_windows.png")
plt.show()
assert backtest.timestamp.max() < holdout_targets.timestamp.min()
assert (folds.train_end_seconds < folds.validation_start_seconds).all()
assert folds.validation_steps.eq(settings["horizon"]).all()


<ipython-input-2-2f5e3d3d3eb1>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Fold,Origin / history rows,History starts (s),History ends (s),Validation starts (s),Validation ends (s),Forecast steps
1,48,0,14100,14400,21300,24
2,54,0,15900,16200,23100,24
3,60,0,17700,18000,24900,24
4,66,0,19500,19800,26700,24
5,72,0,21300,21600,28500,24
6,78,0,23100,23400,30300,24
7,84,0,24900,25200,32100,24


## 4. Keep a single prediction evidence table

**Motive:** Ensure that model comparisons, parameter selection, plots and report tables refer to the same forecast cases.

**Logic:** Store `fold`, `origin`, `step`, `model`, `target`, observed `y` and predicted `yhat`. Keep `timestamp` as an additional alignment field in relative seconds. A unique forecast key is `(fold, step, model, target)`.

Derive fold boundaries, case counts and all performance summaries from this table. Preserve every moving-average candidate rather than replacing its rows with the selected candidate. The complete table is exported as `artifacts/baseline_validation/backtest.csv`.


In [3]:
preview = backtest.loc[backtest.fold.eq(1) & backtest.step.isin([1, 24]) & backtest.model.isin(["persistence", "moving_average_3"])]
display(preview.style.hide(axis="index").format({"y":"{:.5f}", "yhat":"{:.5f}"}))
evidence_checks = pd.DataFrame([
    ("Forecast rows", len(backtest)),
    ("Folds", backtest.fold.nunique()),
    ("Candidate models", backtest.model.nunique()),
    ("Targets", backtest.target.nunique()),
    ("Forecast cases per model and target", int(backtest.groupby(["model", "target"]).size().iloc[0])),
    ("Distinct validation timestamps", backtest.timestamp.nunique()),
    ("Duplicate forecast keys", int(backtest.duplicated(["fold", "step", "model", "target"]).sum())),
], columns=["Measure", "Value"])
display(evidence_checks.style.hide(axis="index").format({"Value":"{:,}"}))
assert not backtest.duplicated(["fold", "step", "model", "target"]).any()
assert np.isfinite(backtest[["y", "yhat"]]).all().all()


fold,origin,step,model,target,timestamp,y,yhat
1,48,1,persistence,active_vm_count,14400,10000.00000,10000.00000
1,48,24,persistence,active_vm_count,21300,10000.00000,10000.00000
1,48,1,moving_average_3,active_vm_count,14400,10000.00000,10000.00000
1,48,24,moving_average_3,active_vm_count,21300,10000.00000,10000.00000
1,48,1,persistence,aggregate_cpu_utilization,14400,9.29989,9.38481
1,48,24,persistence,aggregate_cpu_utilization,21300,9.59058,9.38481
1,48,1,moving_average_3,aggregate_cpu_utilization,14400,9.29989,9.36921
1,48,24,moving_average_3,aggregate_cpu_utilization,21300,9.59058,9.36921


Measure,Value
Forecast rows,"2,352"
Folds,7
Candidate models,7
Targets,2
Forecast cases per model and target,168
Distinct validation timestamps,60
Duplicate forecast keys,0


## 5. Score all candidates using the same cases

**Motive:** Compare average absolute error, sensitivity to larger errors and relative performance against persistence.

**Logic:** Pool the origin/step cases within each target/model pair:

- **MAE:** mean absolute forecast error.
- **RMSE:** square root of mean squared forecast error.
- **sMAPE:** mean `200 × abs(y − yhat) / max(abs(y) + abs(yhat), 1e−8)`.
- **Skill versus persistence:** `1 − MAE_model / MAE_persistence`; positive means lower MAE than persistence.

CPU MAE/RMSE are in percentage points; count MAE/RMSE are in VMs. sMAPE is a percentage. Skill is a dimensionless fraction. If persistence MAE is zero, skill is undefined and shown as unavailable, including a zero-error tie. The metric tables rank by MAE, then RMSE and model name for deterministic display.


In [4]:
scores = score_backtest(backtest)
labels = {"persistence":"Persistence", "drift":"Drift", "seasonal_naive_24":"Seasonal naive (24)",
    **{f"moving_average_{k}": f"Moving average ({k})" for k in (3, 6, 12, 24)}}
for target in ["aggregate_cpu_utilization", "active_vm_count"]:
    table = scores.loc[scores.target.eq(target)].copy()
    table["model"] = table.model.map(labels)
    table = table[["model", "mae", "rmse", "smape_pct", "skill_vs_persistence", "forecast_cases"]]
    table.columns = ["Model", "MAE", "RMSE", "sMAPE (%)", "Skill vs persistence", "Forecast cases"]
    display(table.style.set_caption(target).hide(axis="index").format({"MAE":"{:.5f}", "RMSE":"{:.5f}", "sMAPE (%)":"{:.3f}", "Skill vs persistence":"{:+.3f}"}, na_rep="Undefined: reference MAE = 0"))
fig = plot_baseline_scores(backtest)
save_figure(fig, FIGURES, "baseline_comparison.png")
plt.show()


<ipython-input-4-cb6ef62a993b>:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Model,MAE,RMSE,sMAPE (%),Skill vs persistence,Forecast cases
Moving average (3),0.24385,0.29532,2.561,+0.082,168
Moving average (6),0.25321,0.30507,2.660,+0.046,168
Persistence,0.26550,0.31736,2.783,+0.000,168
Moving average (12),0.26879,0.31207,2.826,-0.012,168
Moving average (24),0.27886,0.31081,2.938,-0.050,168
Drift,0.30469,0.36951,3.182,-0.148,168
Seasonal naive (24),0.32104,0.36111,3.369,-0.209,168


Model,MAE,RMSE,sMAPE (%),Skill vs persistence,Forecast cases
Drift,0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (12),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (24),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (3),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (6),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Persistence,0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Seasonal naive (24),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168


## 6. Select the moving-average window by CV

**Motive:** Choose the smoothing window without using holdout labels.

**Logic:** For each target, select the minimum pooled CV MAE among k = 3, 6, 12 and 24. Break exact ties toward the smaller window. Keep candidate comparisons visible in the preceding table. This selects a parameter on the same CV cases used for comparison; its selected CV score is not an independent final evaluation.

Save the evidence and selected windows for subsequent model comparison. This calculation does not fit a main model, freeze a final forecasting configuration or evaluate the reserved holdout.


In [5]:
selected_ma = select_moving_average(scores)
display(selected_ma[["target", "window", "mae", "rmse", "smape_pct", "skill_vs_persistence"]]
    .rename(columns={"target":"Target", "window":"Selected k", "mae":"CV MAE", "rmse":"CV RMSE", "smape_pct":"CV sMAPE (%)", "skill_vs_persistence":"Skill vs persistence"})
    .style.hide(axis="index").format({"CV MAE":"{:.5f}", "CV RMSE":"{:.5f}", "CV sMAPE (%)":"{:.3f}", "Skill vs persistence":"{:+.3f}"}, na_rep="Undefined: reference MAE = 0"))
for name, table in [("backtest", backtest), ("folds", folds), ("scores", scores), ("selected_moving_average", selected_ma)]:
    table.to_csv(ARTIFACTS / f"{name}.csv", index=False)
summary = {
    "settings": settings,
    "training_rows": len(train_targets),
    "holdout_rows_reserved": len(holdout_targets),
    "holdout_first_timestamp": int(holdout_targets.timestamp.min()),
    "forecast_rows": len(backtest),
    "distinct_validation_timestamps": int(backtest.timestamp.nunique()),
    "folds": json.loads(folds.to_json(orient="records")),
    "scores": json.loads(scores.to_json(orient="records", double_precision=15)),
    "selected_moving_average": json.loads(selected_ma.to_json(orient="records", double_precision=15)),
}
(ARTIFACTS / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")


Out[5]: 6607


Target,Selected k,CV MAE,CV RMSE,CV sMAPE (%),Skill vs persistence
active_vm_count,3,0.00000,0.00000,0.000,Undefined: reference MAE = 0
aggregate_cpu_utilization,3,0.24385,0.29532,2.561,+0.082


## Aggregate, statistical and segment-model fitting

The following original workflow constructs origin-safe predictors and fits SES, ETS, Theta, direct ridge and a shared segment LightGBM on the same cases. The ensemble uses only matured earlier-origin errors among the retained candidates. These experiments support comparison; they do not overwrite the final model configuration.

## 2. Construct features from information available at each origin

**Motive:** Represent recent CPU level, variability, trend, segment mix and observed population changes without exposing future observations.

**Logic:** `make_features(series, origin)` uses an **inclusive last-observed row index**. The backtest origin is a history length, so pass `cv_origin − 1`. Lag 1 is the latest observed value used to forecast the next sample.

| Feature family | Calculation | Latest source row |
|---|---|---|
| CPU lags | Most recent 1–12 observations | Inclusive feature origin |
| Rolling mean / sample SD | Trailing windows of 3, 6 and 12 | Inclusive feature origin |
| Trend slopes | Least-squares slope over trailing 6 and 12 values | Inclusive feature origin |
| Category CPU context | Current category mean and trailing six-sample mean | Inclusive feature origin |
| Category mix | Observed VM share by category and trailing six-sample mean | Inclusive feature origin |
| Population activity | Current observed count, its change, six-step mean change, observed arrivals/departures and six-step totals | Inclusive feature origin |

Arrivals/departures compare observed VM-ID sets with the preceding timestamp; they are not actual creation/deletion events. The first sample has unknown prior activity. Missing **feature inputs** may forward-fill from the past; target labels are never filled. Twelve observations are required before a feature row is usable. Metadata lifetime CPU statistics and deletion timestamps are excluded.


In [6]:
from cwfc.features import make_features, feature_availability
from cwfc.models import (build_model_inputs, LIGHTGBM_PARAMS,
    RIDGE_ALPHA, PANEL_STRIDE)
from cwfc.validation import backtest_models
from cwfc.eda import plot_model_comparison

MODEL_ARTIFACTS = ROOT / "artifacts" / "model_selection"
MODEL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
model_series, segment_panel = build_model_inputs(readings, metadata, train_targets)
first_origin = int(folds.origin.min())
availability = feature_availability(model_series, first_origin - 1)
display(availability.rename(columns={"feature":"Feature", "latest_row_allowed":"Latest allowed row", "latest_timestamp_allowed":"Latest allowed timestamp (s)", "available_at_origin":"Available"})
    .style.hide(axis="index"))
assert availability.available_at_origin.all()
assert availability.latest_timestamp_allowed.max() < int(folds.validation_start_seconds.min())


Feature,Latest allowed row,Latest allowed timestamp (s),Available
lag_1,47,14100,True
lag_2,47,14100,True
lag_3,47,14100,True
lag_4,47,14100,True
lag_5,47,14100,True
lag_6,47,14100,True
lag_7,47,14100,True
lag_8,47,14100,True
lag_9,47,14100,True
lag_10,47,14100,True


## 3. Fit the planned model families

**Motive:** Compare alternatives that make different assumptions about persistence, trend and cross-series information.

| Candidate | Fitting and forecast logic | Fixed choices |
|---|---|---|
| Simple exponential smoothing | Fit a level with exponentially decreasing weights on older observations | Estimated initialization; optimized smoothing |
| Damped-trend ETS | Fit additive level/trend and damp trend extrapolation | No seasonal component |
| Theta | Fit a nonseasonal Theta model | Deseasonalization disabled |
| Direct ridge | Fit one regression per forecast horizon, standardizing predictors inside that horizon's training set | Alpha = 10; no random split |
| Global panel LightGBM | Fit one L1 regression across category/core/memory segment series and horizon steps | Every third training anchor; deterministic seed 42 |

For a historical anchor `t`, a supervised label `y[t+h]` is included only if `t+h` is inside the history available at the CV origin. The standardizer and every model are fitted again within each fold. No validation labels drive early stopping or parameter fitting. Constant statistical-model histories return that constant without numerical optimization. Optimization failures are explicit errors rather than undisclosed substitutions.


In [7]:
model_settings = pd.DataFrame([
    ("Statistical", "SES; damped additive trend ETS; nonseasonal Theta"),
    ("Direct ridge", f"alpha={RIDGE_ALPHA}; one fitted standardizer/regressor per step"),
    ("Panel grouping", "category × core bucket × memory bucket"),
    ("Panel anchor stride", PANEL_STRIDE),
    ("LightGBM objective", LIGHTGBM_PARAMS["objective"]),
    ("LightGBM trees / leaves", f"{LIGHTGBM_PARAMS['n_estimators']} / {LIGHTGBM_PARAMS['num_leaves']}"),
    ("LightGBM learning rate", LIGHTGBM_PARAMS["learning_rate"]),
    ("LightGBM seed / threads", f"{LIGHTGBM_PARAMS['random_state']} / {LIGHTGBM_PARAMS['n_jobs']}"),
], columns=["Setting", "Value"])
display(model_settings.style.hide(axis="index"))


Setting,Value
Statistical,SES; damped additive trend ETS; nonseasonal Theta
Direct ridge,alpha=10.0; one fitted standardizer/regressor per step
Panel grouping,category × core bucket × memory bucket
Panel anchor stride,3
LightGBM objective,regression_l1
LightGBM trees / leaves,120 / 7
LightGBM learning rate,0.050000
LightGBM seed / threads,42 / 1


## 4. Pool segment histories and aggregate bottom-up

**Motive:** Share training examples across related workload series while retaining segment-level predictions.

**Logic:** Construct each observed category/core/memory segment on the training time grid. At each historical anchor, use segment lags, rolling mean/SD/min/max, slopes, observed counts, burstiness (`mean(max_cpu) − mean(min_cpu)`), historical mean, current aggregate CPU context and static segment labels. Include forecast step `h` as a predictor. Only matured, observed segment labels become examples.

This uses **segment series**, one of the panel options in the plan. It does not treat all VM readings as independent training examples or claim a million fitted samples. The fit diagnostics show actual sample counts, including repeated anchors across horizon labels.

Aggregate predicted segment CPU means using the **last observed segment VM counts**, held constant over the forecast horizon. These are population weights, not exact core counts. Missing segment CPU labels stay missing. The CPU aggregation assumes future observed segment mix stays fixed; the separate count fallback does not forecast segment membership.


## 5. Backtest every candidate on the shared folds

**Motive:** Compare models on identical observed outcomes and forecast horizons.

**Logic:** Reuse origins 48, 54, 60, 66, 72, 78 and 84 with 24-step horizons. Models receive a bounded history and panel rows at or before its last timestamp. Add main-model, ensemble and count-fallback rows to the same long-format schema used by the baseline backtest. Keep fit diagnostics separate from performance evidence.


In [8]:
model_result = backtest_models(model_series, segment_panel, **settings)
model_backtest = model_result["backtest"]
# The shared runner must reproduce exactly the previously evaluated references.
pd.testing.assert_frame_equal(backtest, model_backtest.iloc[:len(backtest)].reset_index(drop=True))
model_scores = score_backtest(model_backtest)
main_labels = {**labels, "ses":"Simple exponential smoothing", "ets_damped":"Damped-trend ETS",
    "theta":"Theta", "ridge_direct":"Direct ridge", "global_panel_lightgbm":"Global segment LightGBM",
    "cv_weighted_ensemble":"Past-error weighted ensemble", "count_churn":"Observed-count trend fallback"}
for target in ["aggregate_cpu_utilization", "active_vm_count"]:
    table = model_scores.loc[model_scores.target.eq(target)].copy()
    table["model"] = table.model.map(main_labels)
    display(table[["model", "mae", "rmse", "smape_pct", "skill_vs_persistence", "forecast_cases"]]
        .rename(columns={"model":"Model", "mae":"MAE", "rmse":"RMSE", "smape_pct":"sMAPE (%)", "skill_vs_persistence":"Skill vs persistence", "forecast_cases":"Forecast cases"})
        .style.set_caption(target).hide(axis="index").format({"MAE":"{:.5f}", "RMSE":"{:.5f}", "sMAPE (%)":"{:.3f}", "Skill vs persistence":"{:+.3f}"}, na_rep="Undefined: reference MAE = 0"))
fig = plot_model_comparison(model_backtest)
save_figure(fig, FIGURES, "model_comparison.png")
plt.show()
assert model_backtest.timestamp.max() < holdout_targets.timestamp.min()
assert not model_backtest.duplicated(["fold", "step", "model", "target"]).any()
assert model_result["fit_diagnostics"].latest_label_row.lt(model_result["fit_diagnostics"].origin).all()


<ipython-input-8-d57ef17b3aee>:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Model,MAE,RMSE,sMAPE (%),Skill vs persistence,Forecast cases
Moving average (3),0.24385,0.29532,2.561,+0.082,168
Simple exponential smoothing,0.25312,0.30331,2.658,+0.047,168
Moving average (6),0.25321,0.30507,2.660,+0.046,168
Persistence,0.26550,0.31736,2.783,+0.000,168
Moving average (12),0.26879,0.31207,2.826,-0.012,168
Global segment LightGBM,0.26926,0.30750,2.832,-0.014,168
Theta,0.27424,0.32671,2.878,-0.033,168
Moving average (24),0.27886,0.31081,2.938,-0.050,168
Damped-trend ETS,0.27985,0.33868,2.931,-0.054,168
Past-error weighted ensemble,0.30202,0.34891,3.167,-0.138,168


Model,MAE,RMSE,sMAPE (%),Skill vs persistence,Forecast cases
Observed-count trend fallback,0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Drift,0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (12),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (24),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (3),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Moving average (6),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Persistence,0.00000,0.00000,0.000,Undefined: reference MAE = 0,168
Seasonal naive (24),0.00000,0.00000,0.000,Undefined: reference MAE = 0,168


In [9]:
diagnostics = model_result["fit_diagnostics"]
panel_diagnostics = diagnostics.loc[diagnostics.model.eq("global_panel_lightgbm")]
display(panel_diagnostics[["fold", "origin", "training_samples", "segments", "features", "latest_label_row"]]
    .rename(columns={"fold":"Fold", "origin":"History rows", "training_samples":"Pooled training examples", "segments":"Segment series", "features":"Predictors", "latest_label_row":"Latest label row"})
    .style.hide(axis="index").format({"Pooled training examples":"{:,.0f}", "Segment series":"{:.0f}", "Predictors":"{:.0f}"}))
example_panel = model_result["panel_predictions"]
example_panel = example_panel.loc[example_panel.fold.eq(1) & example_panel.step.isin([1, 24])]
display(example_panel[["segment_id", "step", "yhat", "observed_vm_weight"]].head(12)
    .rename(columns={"segment_id":"Category / core / memory", "step":"Forecast step", "yhat":"Predicted CPU (%)", "observed_vm_weight":"Observed VM weight"})
    .style.hide(axis="index").format({"Predicted CPU (%)":"{:.4f}", "Observed VM weight":"{:,.0f}"}))
reconciled = model_result["panel_predictions"].assign(
    weighted=lambda frame: frame.yhat * frame.observed_vm_weight).groupby(["fold", "step"]).agg(
        numerator=("weighted", "sum"), denominator=("observed_vm_weight", "sum"))
panel_aggregate = model_backtest.loc[model_backtest.model.eq("global_panel_lightgbm")].set_index(["fold", "step"])
assert np.allclose(panel_aggregate.yhat.sort_index(), (reconciled.numerator / reconciled.denominator).sort_index())


Fold,History rows,Pooled training examples,Segment series,Predictors,Latest label row
1,48,"5,508",27,37,47
2,54,"6,804",27,37,53
3,60,"8,100",27,37,59
4,66,"9,396",27,37,65
5,72,"10,692",27,37,71
6,78,"11,988",27,37,77
7,84,"13,284",27,37,83


Category / core / memory,Forecast step,Predicted CPU (%),Observed VM weight
"[""Delay-insensitive"", ""2"", ""2""]",1,9.4428,"2,708"
"[""Delay-insensitive"", ""2"", ""2""]",24,9.4281,"2,708"
"[""Delay-insensitive"", ""2"", ""32""]",1,3.8922,2
"[""Delay-insensitive"", ""2"", ""32""]",24,3.8930,2
"[""Delay-insensitive"", ""2"", ""4""]",1,8.6281,"1,602"
"[""Delay-insensitive"", ""2"", ""4""]",24,8.5320,"1,602"
"[""Delay-insensitive"", ""2"", ""8""]",1,5.0030,580
"[""Delay-insensitive"", ""2"", ""8""]",24,4.9725,580
"[""Delay-insensitive"", ""24"", ""64""]",1,11.7116,34
"[""Delay-insensitive"", ""24"", ""64""]",24,11.6549,34


## 6. Combine forecasts with error information available at the origin

**Motive:** Evaluate inverse-CV-error weighting without using future errors to choose past forecast weights.

**Logic:** At each origin, consider earlier-origin predictions only when their actual timestamp is already observed. Rank candidates by MAE on those matured cases, select up to three and normalize inverse errors to sum to one. If any selected candidate has zero MAE, divide weight equally among zero-error candidates. Before any CV errors exist, use equal weights across all thirteen fixed CPU candidates.

The ensemble CV score uses these sequential weights. A separate set of deployment weights is calculated from all completed training CV cases when the final configuration is frozen. Those deployment weights are not applied retrospectively to improve the ensemble's CV score.


In [10]:
weights = model_result["ensemble_weights"]
display(weights.loc[weights.origin.isin([48, 54, 84])]
    .rename(columns={"fold":"Fold", "origin":"History rows", "model":"Component", "weight":"Weight", "available_error_cases":"Matured cases", "latest_error_timestamp":"Latest error timestamp (s)", "cutoff_timestamp":"Observed cutoff (s)"})
    .style.hide(axis="index").format({"Weight":"{:.4f}", "Latest error timestamp (s)":"{:,.0f}"}, na_rep="No earlier errors"))
known = weights.latest_error_timestamp.notna()
assert (weights.loc[known, "latest_error_timestamp"] <= weights.loc[known, "cutoff_timestamp"]).all()
assert np.allclose(weights.groupby("origin").weight.sum(), 1.)


Fold,History rows,Component,Weight,Matured cases,Latest error timestamp (s),Observed cutoff (s)
1,48,drift,0.0833,0,No earlier errors,14100
1,48,ets_damped,0.0833,0,No earlier errors,14100
1,48,global_panel_lightgbm,0.0833,0,No earlier errors,14100
1,48,moving_average_12,0.0833,0,No earlier errors,14100
1,48,moving_average_24,0.0833,0,No earlier errors,14100
1,48,moving_average_3,0.0833,0,No earlier errors,14100
1,48,moving_average_6,0.0833,0,No earlier errors,14100
1,48,persistence,0.0833,0,No earlier errors,14100
1,48,ridge_direct,0.0833,0,No earlier errors,14100
1,48,seasonal_naive_24,0.0833,0,No earlier errors,14100


## 7. Preserve a simple count model and test the general fallback

**Motive:** Keep persistence as the count reference while allowing a changing observed population in replacement data.

**Logic:** The fallback adds `h × mean(recent observed-count differences)` to the latest count using at most six recent differences, and clips at zero. An unchanged count produces persistence automatically. It uses no future deletion information and cannot distinguish real churn from changing observation coverage.


In [11]:
from cwfc.models import count_churn_forecast
count_examples = []
for name, values in [("Unchanged", [10, 10, 10]), ("Increasing", [10, 12, 14]), ("Decreasing", [4, 2, 0])]:
    prediction = count_churn_forecast(pd.DataFrame({"active_vm_count": values}), 2)
    count_examples.append((name, str(values), prediction[0], prediction[1]))
display(pd.DataFrame(count_examples, columns=["Input pattern", "Observed counts", "Next count", "Following count"])
    .style.hide(axis="index").format({"Next count":"{:.0f}", "Following count":"{:.0f}"}))


Input pattern,Observed counts,Next count,Following count
Unchanged,"[10, 10, 10]",10,10
Increasing,"[10, 12, 14]",16,18
Decreasing,"[4, 2, 0]",0,0


In [12]:
for name, result_table in model_result.items():
    result_table.to_csv(MODEL_ARTIFACTS/f"{name}.csv",index=False)
model_scores.to_csv(MODEL_ARTIFACTS/"scores.csv",index=False)
availability.to_csv(MODEL_ARTIFACTS/"feature_availability.csv",index=False)
display(model_scores.loc[model_scores.target.eq("aggregate_cpu_utilization")].sort_values("mae"))


,target,model,forecast_cases,folds,mae,rmse,smape_pct,persistence_mae,skill_vs_persistence
8,aggregate_cpu_utilization,moving_average_3,168,7,0.243850,0.295324,2.560667,0.265502,0.081550
9,aggregate_cpu_utilization,ses,168,7,0.253123,0.303307,2.658117,0.265502,0.046623
10,aggregate_cpu_utilization,moving_average_6,168,7,0.253212,0.305066,2.660021,0.265502,0.046290
11,aggregate_cpu_utilization,persistence,168,7,0.265502,0.317362,2.783430,0.265502,0.000000
12,aggregate_cpu_utilization,moving_average_12,168,7,0.268795,0.312073,2.826471,0.265502,-0.012403
13,aggregate_cpu_utilization,global_panel_lightgbm,168,7,0.269261,0.307499,2.831738,0.265502,-0.014157
14,aggregate_cpu_utilization,theta,168,7,0.274244,0.326713,2.877590,0.265502,-0.032929
15,aggregate_cpu_utilization,moving_average_24,168,7,0.278856,0.310813,2.937749,0.265502,-0.050298
16,aggregate_cpu_utilization,ets_damped,168,7,0.279849,0.338676,2.930522,0.265502,-0.054037
17,aggregate_cpu_utilization,cv_weighted_ensemble,168,7,0.302017,0.348915,3.166932,0.265502,-0.137534


## Recursive VM LightGBM: inspect the actual training unit

**Motive:** preserve individual histories while sharing one regressor. The 25 predictors are twelve CPU lags, six rolling statistics, two slopes, two context values and three categorical resource/workload labels. The residual label is the next observed CPU minus that VM's past-only level. Missing labels remain missing. The historical level is an internal centering calculation in LightGBM.

The next cell builds the real VM matrices and checks a feature row's information boundary before fitting.

In [13]:
from cwfc.model import MODEL, FEATURES, vm_history, _features, vm_forecast
from cwfc.predict import load_config, run_prediction, validate_predictions
from cwfc.validation import backtest_final, regression_scores
final_config=load_config(ROOT/"configs/final_model.yaml")
values,static,ids,history_info=vm_history(readings,metadata,train_targets.timestamp)
filled=pd.DataFrame(values).ffill(axis=1).to_numpy()
feature_row,vm_level=_features(values,filled,11,static)
display(pd.DataFrame(feature_row[:5],columns=FEATURES))
assert feature_row.shape[1]==25
assert np.allclose(feature_row[:,0],filled[:,11],equal_nan=True)
display(pd.DataFrame([history_info]))


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12,mean_3,std_3,mean_6,std_6,mean_12,std_12,slope_6,slope_12,vm_history_mean,aggregate_cpu_context,vm_category,virtual_core_bucket,memory_gb_bucket
0,54.137592,23.141291,18.035555,21.005510,20.470791,16.126316,18.094217,22.032116,18.783556,17.753559,35.134979,44.666306,31.771479,19.537128,25.486176,14.246060,25.781816,12.232480,5.574226,0.009346,25.781816,9.527111,0.0,0.0,0.0
1,13.701999,14.683646,16.598791,15.857052,15.903960,16.748478,15.968450,18.732700,14.829314,16.642645,14.769694,14.829906,14.994812,1.473251,15.582321,1.176035,15.772220,1.330562,-0.518617,-0.056139,15.772220,9.527111,0.0,2.0,1.0
2,4.349259,3.962684,4.271779,4.498282,4.275675,3.929823,6.220623,4.203555,4.201605,4.020115,4.285316,8.286425,4.194574,0.204525,4.214584,0.223682,4.708762,1.278025,0.026620,-0.157489,4.708762,9.527111,1.0,0.0,0.0
3,2.279475,2.105108,2.228755,2.126843,2.247427,2.164010,2.232462,2.170604,2.260756,2.196021,2.264013,2.912617,2.204446,0.089689,2.191936,0.070220,2.265674,0.211237,0.007208,-0.030326,2.265674,9.527111,0.0,0.0,2.0
4,2.514741,2.538261,2.557374,2.482341,2.473550,2.577935,2.574456,2.536762,2.589891,2.541577,2.578755,2.586505,2.536792,0.021354,2.524033,0.041455,2.546012,0.039361,-0.001337,-0.006179,2.546012,9.527111,0.0,2.0,4.0


,raw_training_readings,valid_training_readings,training_timestamps,vm_series,first_training_timestamp,last_training_timestamp,missing_vm_timestamp_cells
0,1100000,1100000,110,10000,0,32700,0


## Fit the final CPU model and forecast recursively

Fit on all eligible observed next-step labels. Append predicted CPU, rebuild features, and repeat through 24 steps. Individual predictions are averaged only after forecasting. This cell performs the actual final fit and writes the single final CSV.

In [14]:
forecast,manifest=run_prediction(ROOT/"challenge_data", ROOT/"predictions/prediction.csv",
    ROOT/"configs/final_model.yaml", ROOT/"artifacts/final_model")
fit=manifest["fit"]
display(pd.DataFrame(list(fit.items()),columns=["Fitting measure","Value"]))
assert fit["latest_label_row"] < fit["training_timestamps"]
assert not fit["holdout_labels_used"]
display(forecast)


,Fitting measure,Value
0,raw_training_readings,1100000
1,valid_training_readings,1100000
2,training_timestamps,110
3,vm_series,10000
4,first_training_timestamp,0
5,last_training_timestamp,32700
6,missing_vm_timestamp_cells,0
7,training_samples,980000
8,latest_label_row,109
9,features,25


,timestamp,active_vm_count,aggregate_cpu_utilization
0,33000,10000.0,9.253006
1,33300,10000.0,9.176752
2,33600,10000.0,9.131158
3,33900,10000.0,9.207404
4,34200,10000.0,9.170562
5,34500,10000.0,9.156000
6,34800,10000.0,9.195285
7,35100,10000.0,9.207352
8,35400,10000.0,9.216555
9,35700,10000.0,9.223294


## Refit at each chronological validation origin

The final model and persistence receive identical origin/step cases. Every fold reconstructs its history and supervised labels. No holdout CPU value enters this backtest. The fixed final recipe is retained irrespective of rerun scores.

In [15]:
final_backtest,final_diagnostics=backtest_final(readings,metadata,train_targets,
    params=final_config["parameters"],**final_config["validation"])
final_scores=regression_scores(final_backtest)
reference_mae=float(final_scores.loc[final_scores.model.eq("vm_persistence"),"mae"].iloc[0])
final_scores["skill_vs_vm_persistence"]=1-final_scores.mae/reference_mae
final_backtest.to_csv(ROOT/"artifacts/final_model/cpu_backtest.csv",index=False)
final_scores.to_csv(ROOT/"artifacts/final_model/cpu_scores.csv",index=False)
final_diagnostics.to_csv(ROOT/"artifacts/final_model/fit_diagnostics.csv",index=False)
display(final_scores)
display(final_diagnostics)
assert final_diagnostics.latest_label_row.lt(final_diagnostics.origin).all()
assert final_backtest.timestamp.max()<holdout_targets.timestamp.min()


,model,forecast_cases,mae,rmse,bias,smape_pct,underforecast_fraction,skill_vs_vm_persistence
0,global_vm_lightgbm,168,0.198813,0.268890,-0.096026,2.102033,0.541667,0.25118
1,vm_persistence,168,0.265502,0.317362,0.158328,2.783430,0.244048,0.00000


,fold,origin,raw_training_readings,valid_training_readings,training_timestamps,vm_series,first_training_timestamp,last_training_timestamp,missing_vm_timestamp_cells,training_samples,latest_label_row,features,forecast_vm_count,holdout_labels_used
0,1,48,480000,480000,48,10000,0,14100,0,360000,47,25,10000,False
1,2,54,540000,540000,54,10000,0,15900,0,420000,53,25,10000,False
2,3,60,600000,600000,60,10000,0,17700,0,480000,59,25,10000,False
3,4,66,660000,660000,66,10000,0,19500,0,540000,65,25,10000,False
4,5,72,720000,720000,72,10000,0,21300,0,600000,71,25,10000,False
5,6,78,780000,780000,78,10000,0,23100,0,660000,77,25,10000,False
6,7,84,840000,840000,84,10000,0,24900,0,720000,83,25,10000,False


## Fit the observed-count comparator and explain the final count rule

The historical count experiment fits Holt's damped level/trend model. The final export uses the most recent observed valid-VM count. Both give 10,000 here because complete-trace selection removes visible churn. Neither result establishes changing-population accuracy.

In [16]:
from cwfc.count_model import count_forecast
holt_count,count_info=count_forecast(train_targets,24)
display(pd.DataFrame(list(count_info.items()),columns=["Count fit","Value"]))
display(pd.DataFrame({"step":np.arange(1,25),"Holt comparator":holt_count,
    "Final observed-count rule":forecast.active_vm_count}))


,Count fit,Value
0,model,holt_damped_count
1,training_timestamps,110
2,smoothing_settings,"{'smoothing_level': 0.3, 'smoothing_trend': 0...."
3,coefficients_optimized,False
4,initial_level,10000.0
5,initial_trend,0.0
6,fitted_last_level,10000.0
7,fitted_last_trend,0.0
8,training_sse,0.0
9,constant_training_count,True


,step,Holt comparator,Final observed-count rule
0,1,10000.0,10000.0
1,2,10000.0,10000.0
2,3,10000.0,10000.0
3,4,10000.0,10000.0
4,5,10000.0,10000.0
5,6,10000.0,10000.0
6,7,10000.0,10000.0
7,8,10000.0,10000.0
8,9,10000.0,10000.0
9,10,10000.0,10000.0


## Inspect fitted features and reconcile the exported forecast

Feature gains describe tree usage. They do not prove causal importance. The mean of individual predicted CPU at each horizon must reproduce the exported fleet CPU.

In [17]:
importance=pd.read_csv(ROOT/"artifacts/final_model/feature_importance.csv")
display(importance)
individual=pd.read_csv(ROOT/"artifacts/final_model/individual_predictions.csv.gz")
reconciled=individual.groupby("step").predicted_cpu.mean()
np.testing.assert_allclose(reconciled,forecast.aggregate_cpu_utilization,rtol=0,atol=1e-8)
validate_predictions(forecast,holdout_targets.timestamp)
fig,axes=plt.subplots(1,2,figsize=(12,4))
importance.head(12).iloc[::-1].plot.barh(x="feature",y="gain",ax=axes[0],legend=False)
axes[1].plot(forecast.timestamp/300,forecast.aggregate_cpu_utilization,marker="o")
axes[1].set(xlabel="Relative observation index",ylabel="Forecast CPU (%)")
fig.tight_layout();plt.show()


<ipython-input-17-3f26014613f6>:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();plt.show()


,feature,gain
0,vm_history_mean,7.045960e+07
1,lag_1,6.048681e+07
2,slope_12,4.021060e+07
3,mean_12,6.623361e+06
4,mean_3,5.842479e+06
5,lag_2,5.806991e+06
6,lag_12,5.617060e+06
7,mean_6,2.831689e+06
8,lag_4,2.370059e+06
9,lag_10,2.356215e+06


## Direct multi-horizon LightGBM: construct real matured examples

The four controlled variants are base, peaks, history and no-category. Direct residual training includes a horizon feature and weights horizons equally. The next cell constructs the first fold's real CPU/classification examples and verifies the label cutoff. These examples support ablation and activity forecasts; the selected final CPU method remains recursive.

In [18]:
from cwfc.direct import make_panel,direct_examples,VARIANTS,variant_columns
from cwfc.operational import load_recipe,fit_forecasts,backtest as backtest_operational
recipe=load_recipe(ROOT/"configs/operational.yaml")
first_history=train_targets.iloc[:48]
panel=make_panel(readings,metadata,first_history.timestamp)
x,residual,busy,weights,names,diagnostic=direct_examples(panel,24,recipe["direct"]["anchor_stride"],recipe["activity"])
display(pd.DataFrame([diagnostic]))
display(pd.DataFrame({"variant":list(VARIANTS),
    "feature_count":[len(variant_columns(names,v)) for v in VARIANTS]}))
assert len(x)==len(residual)==len(busy)==len(weights)
del x,residual,busy,weights


,training_samples,samples_by_horizon,latest_label_row,anchor_stride,horizon_weighting
0,1080000,"[60000, 60000, 60000, 60000, 60000, 60000, 500...",47,6,equal total weight per horizon


,variant,feature_count
0,direct_base,26
1,direct_peaks,36
2,direct_history,38
3,direct_no_category,37


## Fit direct CPU models and two activity classifiers

The full operational run refits four CPU variants, a busy-probability LightGBM, a whole-window event LightGBM, and the recursive reference at every origin. Calibration uses matured earlier-origin out-of-fold probabilities; event labels require the complete future window. The source experiment used an explicit refit switch for this expensive run, retained here. Training functions, settings, and sample construction remain available rather than being replaced by saved-table viewers.

The saved candidate probabilities and policy decisions are supporting artifacts. The only submission CSV remains `predictions/prediction.csv`.

In [19]:
from cwfc.operational import run as run_operational
RETRAIN_OPERATIONAL=False
if RETRAIN_OPERATIONAL:
    operational_manifest=run_operational(data=ROOT/"challenge_data",config=ROOT/"configs/operational.yaml",
        directory=ROOT/"artifacts/operational",output=ROOT/"artifacts/operational/cpu_forecast.csv",
        activity_output=ROOT/"artifacts/operational/activity_forecast.csv")
else:
    print("Saved operational evidence retained; set RETRAIN_OPERATIONAL=True to refit all direct models/classifiers.")


Saved operational evidence retained; set RETRAIN_OPERATIONAL=True to refit all direct models/classifiers.


## Compare direct variants, busy counts and calibrated probabilities

The retained comparison shows that extra peak features and direct forecasting did not improve pooled CPU MAE over recursive LightGBM. Expected CPU-busy count, thresholded mean forecasts and observed population count answer different questions. The moving-average busy-count baseline outperforms the probability model in this experiment.

In [20]:
for folder,name in [("comparison","cpu_scores"),("comparison","vm_error_distribution"),
    ("comparison","behavior_scores"),("operational","busy_count_scores"),("operational","probability_summary")]:
    print(name)
    display(pd.read_csv(ROOT/f"artifacts/{folder}/{name}.csv"))


cpu_scores
vm_error_distribution
behavior_scores
busy_count_scores
probability_summary


,model,forecast_cases,mae,rmse,bias,smape_pct,underforecast_fraction,skill_vs_vm_persistence
0,global_vm_lightgbm,168,0.198813,0.268890,-0.096026,2.102033,0.541667,0.251180
1,vm_moving_average_3,168,0.243850,0.295324,0.071621,2.560667,0.351190,0.081550
2,direct_history,168,0.264530,0.300957,0.070795,2.788571,0.309524,0.003659
3,vm_persistence,168,0.265502,0.317362,0.158328,2.783430,0.244048,0.000000
4,direct_no_category,168,0.265935,0.302153,0.070647,2.803107,0.309524,-0.001632
5,direct_base,168,0.266571,0.299940,0.085999,2.808542,0.303571,-0.004028
6,direct_peaks,168,0.267583,0.300197,0.089173,2.818520,0.303571,-0.007839


,model,vm_origin_cases,macro_mae,median_vm_mae,p90_vm_mae,p99_vm_mae
0,direct_base,70000,2.358361,0.933678,4.692452,25.559503
1,direct_history,70000,2.343023,0.919166,4.633821,25.797602
2,direct_no_category,70000,2.344054,0.915920,4.648513,25.738414
3,direct_peaks,70000,2.351139,0.931629,4.663666,25.610453
4,global_vm_lightgbm,70000,2.133217,0.769886,4.493806,26.165843
5,vm_moving_average_3,70000,2.311259,0.842964,4.626291,27.843306
6,vm_persistence,70000,2.541035,0.883226,5.172461,29.265819


,model,behavior,cases,sum_absolute_error,sum_square_error,sum_error,mae,rmse,bias
0,direct_base,low_average_bursty,17640,5.784692e+04,1.395731e+06,-3051.076037,3.279304,8.895116,-0.172963
1,direct_base,steady,767592,2.003362e+06,2.246031e+07,46204.086960,2.609931,5.409320,0.060194
2,direct_base,sustained_low_recent,28056,1.567524e+04,1.560957e+05,7010.067641,0.558713,2.358754,0.249860
3,direct_base,variable,866712,1.885162e+06,4.147853e+07,94316.049502,2.175073,6.917900,0.108821
4,direct_history,low_average_bursty,17640,5.463867e+04,1.401445e+06,-8486.202973,3.097430,8.913303,-0.481077
5,direct_history,steady,767592,1.999858e+06,2.257134e+07,58380.496359,2.605367,5.422673,0.076057
6,direct_history,sustained_low_recent,28056,1.535295e+04,1.512138e+05,6628.741924,0.547225,2.321576,0.236268
7,direct_history,variable,866712,1.866428e+06,4.126373e+07,62412.636335,2.153458,6.899964,0.072011
8,direct_no_category,low_average_bursty,17640,5.459789e+04,1.390289e+06,-8652.350857,3.095119,8.877758,-0.490496
9,direct_no_category,steady,767592,2.000970e+06,2.261210e+07,56252.087397,2.606814,5.427567,0.073284


,model,forecast_cases,mae,rmse,bias,smape_pct,underforecast_fraction,threshold
0,busy_count_ma3,168,1.351190,1.683840,-0.113095,0.013548,0.428571,0.1
1,busy_count_persistence,168,1.684524,2.065879,-1.208333,0.016890,0.672619,0.1
2,cpu_threshold_count,168,21.232143,21.304622,21.232143,0.212657,0.000000,0.1
3,busy_count_ma3,168,11.565476,14.084143,-0.494048,0.125171,0.505952,1.0
4,busy_probability_model,168,14.041272,17.539427,-2.426898,0.151995,0.500000,1.0
5,busy_count_persistence,168,14.220238,17.645821,8.315476,0.153862,0.327381,1.0
6,cpu_threshold_count,168,313.208333,314.626356,313.208333,3.333057,0.000000,1.0
7,busy_count_ma3,168,52.101190,60.936133,23.065476,1.018859,0.345238,5.0
8,busy_count_persistence,168,86.244048,99.204125,79.113095,1.679670,0.113095,5.0
9,cpu_threshold_count,168,375.059524,380.702937,375.059524,7.090755,0.000000,5.0


,target,stream,cases,sum_brier,sum_log_loss,brier_score,log_loss
0,busy_at_horizon,raw,1680000,22725.770575,76952.433948,0.013527,0.045805
1,busy_at_horizon,served,1680000,22771.689579,77381.396918,0.013555,0.046060
2,two_hour_activity_given_screen,raw,1169,270.157205,833.637057,0.231101,0.713120
3,two_hour_activity_given_screen,served,1169,260.859247,800.215385,0.223147,0.684530


## Final model choice and validation boundary

The final CPU model is recursive VM LightGBM. Selection used pooled chronological CV MAE with RMSE/model-name tie breaks. The holdout was already viewed before this model's experiment; no independent final-model holdout result is claimed. The direct variants, Holt count and activity classifiers are comparison or operational experiments rather than additional final submissions.